# Bilibili 播放量回归基准模型

这个 notebook 用于先在 **20-100 条视频的小规模数据集** 上验证完整训练链路，目标是预测 `log1p(stat_view)`。

模型结构对应当前基准设想：

- `VideoMAE`：读取视频前 15 秒的固定间隔帧，提取视频表征后做 mean pooling。
- `BERT`：对评论文本做编码，再拆成“语义投影”和“情感投影”后横向拼接。
- `Metadata MLP`：对标准化后的元数据做一层小型 MLP。
- `Fusion MLP`：将视频、评论、元数据三路特征拼接后做回归。

当前 notebook 的设计原则：

- **优先支持 SQLite**：直接读取当前爬虫输出的 `videos` / `video_stat_snapshots` / `topn_comment_snapshots` / `assets`。
- **兼容扁平表**：如果你后续在 Colab 里导出成 `csv/parquet/jsonl`，也可以直接读。
- **优先打通链路，不追求最优性能**：默认建议先冻结 `VideoMAE` 和 `BERT` 主干，只训练融合层。
- **允许媒体缺失时先调试文本+元数据**：如果当前样本里没有可直接读取的视频，可以先把 `require_video=False`，验证训练代码和数据流。

> 注意：如果你手头只有 `video_pool` CSV，它通常只包含候选视频列表，不包含真实播放量、评论和媒体路径，因此**不能单独用于这个基准训练**，最多只能作为候选样本列表或补充字段来源。

In [ ]:
# 如果你在 Colab 中运行，先执行这一格安装依赖；如果环境里已经有这些包，可以直接跳过。
%pip -q install -U transformers decord opencv-python-headless pyarrow

from __future__ import annotations

import copy
import io
import json
import math
import os
import random
import sqlite3
import textwrap
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, Video, display
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoTokenizer, BertModel, VideoMAEModel

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print(f"Using device: {DEVICE}")


@dataclass
class BaselineConfig:
    data_source: str = "sqlite"  # sqlite / flatfile
    sqlite_path: Path = Path("/content/bili_video_data_crawler.db")
    flatfile_path: Path = Path("/content/dataset.jsonl")
    candidate_pool_path: Path | None = Path("/content/video_pool.csv")

    sample_size: int | None = 32
    require_video: bool = True

    clip_seconds: int = 15
    num_frames: int = 16
    image_size: int = 224

    bert_model_name: str = "bert-base-chinese"
    videomae_model_name: str = "MCG-NJU/videomae-base"
    text_max_length: int = 256

    batch_size: int = 2
    num_workers: int = 0
    epochs: int = 5
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4

    train_ratio: float = 0.7
    val_ratio: float = 0.15

    freeze_video_backbone: bool = True
    freeze_text_backbone: bool = True

    max_comments: int = 10
    max_comment_chars: int = 48

    use_post_publication_stats: bool = True
    materialized_video_dir: Path = Path("./materialized_videos")

    numeric_metadata_columns: list[str] = field(default_factory=lambda: [
        "duration_seconds",
        "videos_count",
        "tid",
        "tid_v2",
        "owner_mid",
        "owner_level",
        "owner_vip_type",
        "owner_follower_count",
        "owner_following_count",
        "owner_video_count",
        "resolution_width",
        "resolution_height",
        "resolution_rotate",
        "is_activity_participant",
        "is_story",
        "is_interactive_video",
        "is_downloadable",
        "is_reprint_allowed",
        "is_collaboration",
        "is_360",
        "is_paid_video",
        "comment_count",
        "comment_like_mean",
        "comment_len_mean",
        "days_since_pub",
        "has_local_video",
        "has_sqlite_video_asset",
        "seed_score",
        "stat_like",
        "stat_coin",
        "stat_favorite",
        "stat_share",
        "stat_reply",
        "stat_danmu",
        "stat_dislike",
        "stat_his_rank",
        "stat_now_rank",
    ])


config = BaselineConfig()

In [ ]:
def maybe_json_loads(value: Any) -> Any:
    if isinstance(value, (dict, list)):
        return value
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        return value


def read_flat_table(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        return pd.read_json(path)
    raise ValueError(f"Unsupported flat file format: {path}")


def load_sqlite_training_frame(sqlite_path: str | Path) -> pd.DataFrame:
    sqlite_path = Path(sqlite_path)
    if not sqlite_path.exists():
        raise FileNotFoundError(f"SQLite file not found: {sqlite_path}")

    query = """
    WITH latest_stats AS (
        SELECT * FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY bvid ORDER BY snapshot_time DESC, id DESC) AS rn
            FROM video_stat_snapshots
        ) WHERE rn = 1
    ),
    latest_comments AS (
        SELECT * FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY bvid ORDER BY snapshot_time DESC, id DESC) AS rn
            FROM topn_comment_snapshots
        ) WHERE rn = 1
    ),
    latest_video_assets AS (
        SELECT * FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY bvid ORDER BY uploaded_at DESC, id DESC) AS rn
            FROM assets
            WHERE asset_type = 'video'
        ) WHERE rn = 1
    )
    SELECT
        v.*,
        s.snapshot_time AS stat_snapshot_time,
        s.stat_view,
        s.stat_like,
        s.stat_coin,
        s.stat_favorite,
        s.stat_share,
        s.stat_reply,
        s.stat_danmu,
        s.stat_dislike,
        s.stat_his_rank,
        s.stat_now_rank,
        s.stat_evaluation,
        c.snapshot_time AS comment_snapshot_time,
        c.limit_n,
        c.items_json,
        a.storage_backend AS video_storage_backend,
        a.object_key AS video_object_key,
        a.object_url AS video_object_url,
        a.mime_type AS video_mime_type,
        a.file_size AS video_file_size
    FROM videos v
    LEFT JOIN latest_stats s ON v.bvid = s.bvid
    LEFT JOIN latest_comments c ON v.bvid = c.bvid
    LEFT JOIN latest_video_assets a ON v.bvid = a.bvid
    """

    with sqlite3.connect(sqlite_path) as conn:
        df = pd.read_sql_query(query, conn)
    return df


def guess_local_video_path(row: pd.Series) -> str | None:
    candidate_keys = [
        "video_path",
        "local_video_path",
        "video_file",
        "video_object_key",
        "video_object_url",
    ]
    for key in candidate_keys:
        candidate = row.get(key)
        if candidate is None:
            continue
        text = str(candidate).strip()
        if not text:
            continue
        if text.startswith("http://") or text.startswith("https://"):
            continue
        path = Path(text)
        if path.exists():
            return str(path)
    return None


def materialize_sqlite_video_asset(
    sqlite_path: str | Path,
    bvid: str,
    cid: int | None = None,
    asset_type: str = "video",
    output_dir: str | Path = "./materialized_videos",
) -> Path | None:
    sqlite_path = Path(sqlite_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    with sqlite3.connect(sqlite_path) as conn:
        conn.row_factory = sqlite3.Row
        if cid is None:
            asset_row = conn.execute(
                """
                SELECT id, object_key, mime_type
                FROM assets
                WHERE bvid = ? AND asset_type = ?
                ORDER BY uploaded_at DESC, id DESC
                LIMIT 1
                """,
                (bvid, asset_type),
            ).fetchone()
        else:
            asset_row = conn.execute(
                """
                SELECT id, object_key, mime_type
                FROM assets
                WHERE bvid = ? AND cid = ? AND asset_type = ?
                ORDER BY uploaded_at DESC, id DESC
                LIMIT 1
                """,
                (bvid, int(cid), asset_type),
            ).fetchone()
            if asset_row is None:
                asset_row = conn.execute(
                    """
                    SELECT id, object_key, mime_type
                    FROM assets
                    WHERE bvid = ? AND asset_type = ?
                    ORDER BY uploaded_at DESC, id DESC
                    LIMIT 1
                    """,
                    (bvid, asset_type),
                ).fetchone()

        if asset_row is None:
            return None

        object_key = str(asset_row["object_key"] or "")
        suffix = Path(object_key).suffix or ".m4s"
        output_path = output_dir / f"{bvid}__{asset_type}{suffix}"
        if output_path.exists() and output_path.stat().st_size > 0:
            return output_path

        chunks = conn.execute(
            """
            SELECT chunk_data
            FROM asset_chunks
            WHERE asset_id = ?
            ORDER BY chunk_index
            """,
            (int(asset_row["id"]),),
        ).fetchall()
        if not chunks:
            return None

        data = b"".join(chunk["chunk_data"] for chunk in chunks)
        output_path.write_bytes(data)
        return output_path


def comments_to_text(items: Any, max_comments: int = 10, max_comment_chars: int = 48) -> str:
    items = maybe_json_loads(items) or []
    texts: list[str] = []
    for item in items[:max_comments]:
        message = str(item.get("message", "")).replace("\n", " ").strip()
        if not message:
            continue
        texts.append(message[:max_comment_chars])
    return " [SEP] ".join(texts) if texts else "无评论"


def comment_statistics(items: Any) -> dict[str, float]:
    items = maybe_json_loads(items) or []
    likes = [float(item.get("like", 0) or 0) for item in items]
    lengths = [len(str(item.get("message", "")).strip()) for item in items if str(item.get("message", "")).strip()]
    return {
        "comment_count": float(len(items)),
        "comment_like_mean": float(np.mean(likes)) if likes else 0.0,
        "comment_len_mean": float(np.mean(lengths)) if lengths else 0.0,
    }


def normalize_training_frame(df: pd.DataFrame, config: BaselineConfig) -> tuple[pd.DataFrame, list[str]]:
    df = df.copy()

    if config.candidate_pool_path and Path(config.candidate_pool_path).exists() and "bvid" in df.columns:
        pool_df = pd.read_csv(config.candidate_pool_path)
        merge_cols = [col for col in ["bvid", "source_type", "source_ref", "seed_score"] if col in pool_df.columns]
        if merge_cols:
            df = df.merge(pool_df[merge_cols].drop_duplicates(subset=["bvid"]), on="bvid", how="left")

    if "stat_view" not in df.columns:
        for fallback_col in ["view", "play", "play_count"]:
            if fallback_col in df.columns:
                df["stat_view"] = df[fallback_col]
                break

    if "stat_view" not in df.columns:
        raise ValueError("The dataset must contain `stat_view` (or a compatible play-count column).")

    for col in [
        "pubdate",
        "stat_snapshot_time",
        "comment_snapshot_time",
    ]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    if "pubdate" in df.columns:
        now_ts = pd.Timestamp.utcnow().tz_localize(None)
        df["days_since_pub"] = (now_ts - df["pubdate"].dt.tz_localize(None)).dt.total_seconds() / 86400.0
    else:
        df["days_since_pub"] = 0.0

    if "items_json" not in df.columns and "comment_text" in df.columns:
        df["items_json"] = df["comment_text"].fillna("").apply(lambda x: [{"message": str(x), "like": 0}])

    df["items_json"] = df.get("items_json", pd.Series([[]] * len(df))).apply(lambda x: maybe_json_loads(x) or [])
    df["comment_text"] = df["items_json"].apply(
        lambda x: comments_to_text(x, max_comments=config.max_comments, max_comment_chars=config.max_comment_chars)
    )

    comment_stats = df["items_json"].apply(comment_statistics).apply(pd.Series)
    for col in comment_stats.columns:
        df[col] = comment_stats[col]

    df["video_path"] = df.apply(guess_local_video_path, axis=1)
    df["has_local_video"] = df["video_path"].notna().astype(float)
    df["has_sqlite_video_asset"] = (
        (df.get("video_storage_backend", pd.Series([""] * len(df))).fillna("") == "sqlite")
        | pd.to_numeric(df.get("video_file_size", pd.Series([0] * len(df))), errors="coerce").fillna(0).gt(0)
    ).astype(float)

    bool_like_cols = [
        "is_activity_participant",
        "is_story",
        "is_interactive_video",
        "is_downloadable",
        "is_reprint_allowed",
        "is_collaboration",
        "is_360",
        "is_paid_video",
        "owner_verified",
    ]
    for col in bool_like_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(float)

    for col in config.numeric_metadata_columns + ["stat_view"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if not config.use_post_publication_stats:
        for col in [
            "stat_like",
            "stat_coin",
            "stat_favorite",
            "stat_share",
            "stat_reply",
            "stat_danmu",
            "stat_dislike",
            "stat_his_rank",
            "stat_now_rank",
        ]:
            if col in df.columns:
                df[col] = np.nan

    df = df[df["stat_view"].notna() & (df["stat_view"] >= 0)].copy()
    df["log_view"] = np.log1p(df["stat_view"].astype(float))

    if config.require_video:
        has_video = df["has_local_video"].eq(1.0) | df["has_sqlite_video_asset"].eq(1.0)
        df = df[has_video].copy()

    df["sample_id"] = df.apply(
        lambda row: f"{row['bvid']}__{int(row['cid']) if pd.notna(row.get('cid')) else 'na'}",
        axis=1,
    )

    if config.sample_size is not None and len(df) > config.sample_size:
        df = df.sample(config.sample_size, random_state=SEED).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    metadata_cols = [col for col in config.numeric_metadata_columns if col in df.columns]
    return df, metadata_cols


def load_experiment_frame(config: BaselineConfig) -> tuple[pd.DataFrame, list[str]]:
    if config.data_source == "sqlite":
        raw_df = load_sqlite_training_frame(config.sqlite_path)
    elif config.data_source == "flatfile":
        raw_df = read_flat_table(config.flatfile_path)
    else:
        raise ValueError(f"Unsupported data_source: {config.data_source}")

    frame, metadata_cols = normalize_training_frame(raw_df, config)
    print(f"Loaded {len(frame)} samples.")
    print(f"Metadata columns ({len(metadata_cols)}): {metadata_cols}")
    return frame, metadata_cols

In [ ]:
@dataclass
class MetadataScaler:
    columns: list[str]
    fill_values: dict[str, float]
    means: dict[str, float]
    stds: dict[str, float]

    @classmethod
    def fit(cls, df: pd.DataFrame, columns: list[str]) -> "MetadataScaler":
        work_df = df.copy()
        if not columns:
            work_df["__dummy__"] = 0.0
            columns = ["__dummy__"]

        fill_values: dict[str, float] = {}
        means: dict[str, float] = {}
        stds: dict[str, float] = {}
        for col in columns:
            values = pd.to_numeric(work_df[col], errors="coerce") if col in work_df.columns else pd.Series(0.0, index=work_df.index)
            fill = float(values.median()) if values.notna().any() else 0.0
            filled = values.fillna(fill).astype(float)
            std = float(filled.std())
            fill_values[col] = fill
            means[col] = float(filled.mean())
            stds[col] = std if std > 1e-6 else 1.0
        return cls(columns=columns, fill_values=fill_values, means=means, stds=stds)

    def transform_row(self, row: pd.Series) -> np.ndarray:
        values = []
        for col in self.columns:
            value = row[col] if col in row.index else np.nan
            value = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
            if pd.isna(value):
                value = self.fill_values[col]
            value = (float(value) - self.means[col]) / self.stds[col]
            values.append(value)
        return np.asarray(values, dtype=np.float32)


def split_dataframe(
    df: pd.DataFrame,
    train_ratio: float = 0.7,
    val_ratio: float = 0.15,
    seed: int = SEED,
) -> dict[str, pd.DataFrame]:
    if len(df) < 3:
        raise ValueError("Need at least 3 samples to build train/val/test splits.")

    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(df))

    n_train = max(1, int(len(df) * train_ratio))
    n_val = max(1, int(len(df) * val_ratio))
    n_test = len(df) - n_train - n_val
    if n_test <= 0:
        n_test = 1
        n_train = max(1, n_train - 1)

    train_idx = indices[:n_train]
    val_idx = indices[n_train : n_train + n_val]
    test_idx = indices[n_train + n_val :]
    if len(test_idx) == 0:
        test_idx = val_idx[-1:]
        val_idx = val_idx[:-1]

    return {
        "train": df.iloc[train_idx].reset_index(drop=True),
        "val": df.iloc[val_idx].reset_index(drop=True),
        "test": df.iloc[test_idx].reset_index(drop=True),
    }


def resolve_video_source(row: pd.Series, config: BaselineConfig) -> str | None:
    local_path = guess_local_video_path(row)
    if local_path is not None:
        return local_path
    if config.data_source == "sqlite" and config.sqlite_path.exists():
        materialized = materialize_sqlite_video_asset(
            sqlite_path=config.sqlite_path,
            bvid=str(row.get("bvid")),
            cid=int(row["cid"]) if pd.notna(row.get("cid")) else None,
            asset_type="video",
            output_dir=config.materialized_video_dir,
        )
        if materialized is not None and materialized.exists():
            return str(materialized)
    return None


def sample_video_frames(
    video_path: str | None,
    clip_seconds: int,
    num_frames: int,
    image_size: int,
) -> np.ndarray:
    blank = np.zeros((num_frames, image_size, image_size, 3), dtype=np.uint8)
    if not video_path:
        return blank

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return blank

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 0 or math.isnan(fps):
        fps = 25.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration_seconds = frame_count / fps if frame_count > 0 else clip_seconds
    effective_seconds = max(0.2, min(float(clip_seconds), float(duration_seconds)))
    timestamps = np.linspace(0.0, effective_seconds, num_frames, endpoint=False)

    frames: list[np.ndarray] = []
    last_valid: np.ndarray | None = None
    for timestamp in timestamps:
        cap.set(cv2.CAP_PROP_POS_MSEC, float(timestamp) * 1000.0)
        ok, frame = cap.read()
        if ok and frame is not None:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            last_valid = frame
        elif last_valid is not None:
            frame = last_valid.copy()
        else:
            frame = np.zeros((image_size, image_size, 3), dtype=np.uint8)
        frames.append(frame)

    cap.release()
    return np.stack(frames)


class BiliPopularityDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        metadata_scaler: MetadataScaler,
        tokenizer: AutoTokenizer,
        image_processor: AutoImageProcessor,
        config: BaselineConfig,
    ) -> None:
        self.frame = frame.reset_index(drop=True)
        self.metadata_scaler = metadata_scaler
        self.tokenizer = tokenizer
        self.image_processor = image_processor
        self.config = config

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> dict[str, Any]:
        row = self.frame.iloc[index]
        metadata = self.metadata_scaler.transform_row(row)

        text_inputs = self.tokenizer(
            str(row.get("comment_text", "无评论")),
            max_length=self.config.text_max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        video_path = resolve_video_source(row, self.config)
        frames = sample_video_frames(
            video_path=video_path,
            clip_seconds=self.config.clip_seconds,
            num_frames=self.config.num_frames,
            image_size=self.config.image_size,
        )
        video_inputs = self.image_processor(list(frames), return_tensors="pt")
        pixel_values = video_inputs["pixel_values"]
        if pixel_values.ndim == 5:
            pixel_values = pixel_values.squeeze(0)

        return {
            "pixel_values": pixel_values,
            "input_ids": text_inputs["input_ids"].squeeze(0),
            "attention_mask": text_inputs["attention_mask"].squeeze(0),
            "metadata": torch.tensor(metadata, dtype=torch.float32),
            "label": torch.tensor(float(row["log_view"]), dtype=torch.float32),
            "bvid": str(row.get("bvid", "")),
            "title": str(row.get("title", "")),
            "video_path": video_path or "",
        }


class MultiModalRegressor(nn.Module):
    def __init__(self, config: BaselineConfig, metadata_dim: int) -> None:
        super().__init__()
        self.video_backbone = VideoMAEModel.from_pretrained(config.videomae_model_name)
        self.text_backbone = BertModel.from_pretrained(config.bert_model_name)

        if config.freeze_video_backbone:
            for param in self.video_backbone.parameters():
                param.requires_grad = False
        if config.freeze_text_backbone:
            for param in self.text_backbone.parameters():
                param.requires_grad = False

        video_hidden = self.video_backbone.config.hidden_size
        text_hidden = self.text_backbone.config.hidden_size

        self.video_proj = nn.Sequential(
            nn.LayerNorm(video_hidden),
            nn.Linear(video_hidden, 256),
            nn.GELU(),
            nn.Dropout(0.1),
        )
        self.text_semantic_proj = nn.Sequential(
            nn.LayerNorm(text_hidden),
            nn.Linear(text_hidden, 128),
            nn.GELU(),
        )
        self.text_sentiment_proj = nn.Sequential(
            nn.LayerNorm(text_hidden),
            nn.Linear(text_hidden, 128),
            nn.Tanh(),
        )
        self.metadata_mlp = nn.Sequential(
            nn.Linear(metadata_dim, 128),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.GELU(),
        )
        self.fusion_mlp = nn.Sequential(
            nn.Linear(256 + 128 + 128 + 64, 256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Linear(64, 1),
        )

    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        metadata: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        video_outputs = self.video_backbone(pixel_values=pixel_values)
        video_feature = video_outputs.last_hidden_state.mean(dim=1)
        video_feature = self.video_proj(video_feature)

        text_outputs = self.text_backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled_text = text_outputs.pooler_output
        if pooled_text is None:
            pooled_text = text_outputs.last_hidden_state[:, 0]
        text_semantic = self.text_semantic_proj(pooled_text)
        text_sentiment = self.text_sentiment_proj(pooled_text)

        metadata_feature = self.metadata_mlp(metadata)
        fused = torch.cat([video_feature, text_semantic, text_sentiment, metadata_feature], dim=-1)
        prediction = self.fusion_mlp(fused).squeeze(-1)
        return {
            "prediction": prediction,
            "video_feature": video_feature,
            "text_feature": torch.cat([text_semantic, text_sentiment], dim=-1),
            "metadata_feature": metadata_feature,
        }


def move_batch_to_device(batch: dict[str, Any], device: torch.device) -> dict[str, Any]:
    moved: dict[str, Any] = {}
    for key, value in batch.items():
        if torch.is_tensor(value):
            moved[key] = value.to(device)
        else:
            moved[key] = value
    return moved

In [ ]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    mse = float(np.mean((y_true - y_pred) ** 2))
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(mse))
    denom = float(np.sum((y_true - y_true.mean()) ** 2))
    r2 = 1.0 - float(np.sum((y_true - y_pred) ** 2)) / (denom + 1e-8)
    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    }


def build_dataloaders(
    splits: dict[str, pd.DataFrame],
    metadata_scaler: MetadataScaler,
    tokenizer: AutoTokenizer,
    image_processor: AutoImageProcessor,
    config: BaselineConfig,
) -> tuple[dict[str, BiliPopularityDataset], dict[str, DataLoader]]:
    datasets = {
        split_name: BiliPopularityDataset(
            frame=split_df,
            metadata_scaler=metadata_scaler,
            tokenizer=tokenizer,
            image_processor=image_processor,
            config=config,
        )
        for split_name, split_df in splits.items()
    }

    loaders = {
        split_name: DataLoader(
            dataset,
            batch_size=config.batch_size,
            shuffle=(split_name == "train"),
            num_workers=config.num_workers,
        )
        for split_name, dataset in datasets.items()
    }
    return datasets, loaders


def run_epoch(
    model: MultiModalRegressor,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer | None,
    device: torch.device,
) -> tuple[float, dict[str, float]]:
    train_mode = optimizer is not None
    model.train(train_mode)
    criterion = nn.MSELoss()

    total_loss = 0.0
    y_true: list[float] = []
    y_pred: list[float] = []

    for batch in tqdm(loader, leave=False):
        batch = move_batch_to_device(batch, device)
        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        outputs = model(
            pixel_values=batch["pixel_values"],
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            metadata=batch["metadata"],
        )
        loss = criterion(outputs["prediction"], batch["label"])

        if train_mode:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        total_loss += float(loss.item()) * len(batch["label"])
        y_true.extend(batch["label"].detach().cpu().numpy().tolist())
        y_pred.extend(outputs["prediction"].detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(len(loader.dataset), 1)
    metrics = regression_metrics(np.array(y_true), np.array(y_pred))
    metrics["loss"] = avg_loss
    return avg_loss, metrics


def train_model(
    model: MultiModalRegressor,
    loaders: dict[str, DataLoader],
    config: BaselineConfig,
    device: torch.device = DEVICE,
) -> tuple[MultiModalRegressor, pd.DataFrame]:
    model = model.to(device)
    optimizer = torch.optim.AdamW(
        [param for param in model.parameters() if param.requires_grad],
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )

    history: list[dict[str, float]] = []
    best_state = copy.deepcopy(model.state_dict())
    best_val_rmse = float("inf")

    for epoch in range(1, config.epochs + 1):
        train_loss, train_metrics = run_epoch(model, loaders["train"], optimizer, device)
        with torch.no_grad():
            _, val_metrics = run_epoch(model, loaders["val"], optimizer=None, device=device)

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_rmse": train_metrics["rmse"],
            "train_mae": train_metrics["mae"],
            "train_r2": train_metrics["r2"],
            "val_loss": val_metrics["loss"],
            "val_rmse": val_metrics["rmse"],
            "val_mae": val_metrics["mae"],
            "val_r2": val_metrics["r2"],
        }
        history.append(row)
        print(pd.Series(row).to_string())
        print("-" * 60)

        if val_metrics["rmse"] < best_val_rmse:
            best_val_rmse = val_metrics["rmse"]
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history)
    return model, history_df


def evaluate_model(
    model: MultiModalRegressor,
    loader: DataLoader,
    device: torch.device = DEVICE,
) -> tuple[pd.DataFrame, dict[str, float]]:
    model.eval()
    y_true: list[float] = []
    y_pred: list[float] = []
    rows: list[dict[str, Any]] = []

    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            meta = {
                "bvid": batch["bvid"],
                "title": batch["title"],
                "video_path": batch["video_path"],
            }
            batch = move_batch_to_device(batch, device)
            outputs = model(
                pixel_values=batch["pixel_values"],
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                metadata=batch["metadata"],
            )
            preds = outputs["prediction"].detach().cpu().numpy()
            labels = batch["label"].detach().cpu().numpy()

            for idx in range(len(preds)):
                rows.append(
                    {
                        "bvid": meta["bvid"][idx],
                        "title": meta["title"][idx],
                        "video_path": meta["video_path"][idx],
                        "target_log_view": float(labels[idx]),
                        "pred_log_view": float(preds[idx]),
                        "target_view": float(np.expm1(labels[idx])),
                        "pred_view": float(np.expm1(preds[idx])),
                    }
                )
            y_true.extend(labels.tolist())
            y_pred.extend(preds.tolist())

    prediction_df = pd.DataFrame(rows)
    metrics = regression_metrics(np.asarray(y_true), np.asarray(y_pred))
    return prediction_df, metrics

In [ ]:
# 1. 先根据你的数据位置修改 config。
# 2. 再运行这一格，完成数据读取、切分、训练和测试集评估。

experiment_df, metadata_columns = load_experiment_frame(config)

if len(experiment_df) < 3:
    raise ValueError("当前可用样本少于 3 条，无法构建 train/val/test。")

preview_cols = [
    col
    for col in [
        "sample_id",
        "bvid",
        "title",
        "stat_view",
        "log_view",
        "duration_seconds",
        "comment_count",
        "has_local_video",
        "has_sqlite_video_asset",
        "video_path",
    ]
    if col in experiment_df.columns
]
display(experiment_df[preview_cols].head())

splits = split_dataframe(
    experiment_df,
    train_ratio=config.train_ratio,
    val_ratio=config.val_ratio,
    seed=SEED,
)
for split_name, split_df in splits.items():
    print(f"{split_name}: {len(split_df)} samples")

metadata_scaler = MetadataScaler.fit(splits["train"], metadata_columns)
tokenizer = AutoTokenizer.from_pretrained(config.bert_model_name)
image_processor = AutoImageProcessor.from_pretrained(config.videomae_model_name)

datasets, loaders = build_dataloaders(
    splits=splits,
    metadata_scaler=metadata_scaler,
    tokenizer=tokenizer,
    image_processor=image_processor,
    config=config,
)

model = MultiModalRegressor(config=config, metadata_dim=len(metadata_scaler.columns))
trained_model, history_df = train_model(model=model, loaders=loaders, config=config, device=DEVICE)
display(history_df)

test_predictions, test_metrics = evaluate_model(trained_model, loaders["test"], device=DEVICE)
print("Test metrics:")
print(pd.Series(test_metrics).to_string())
display(test_predictions.sort_values("target_view", ascending=False).head(10))

In [ ]:
def show_frame_grid(frames: np.ndarray, cols: int = 4, figsize: tuple[int, int] = (16, 8)) -> None:
    cols = max(1, cols)
    rows = math.ceil(len(frames) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(rows, cols)
    for ax in axes.flat:
        ax.axis("off")
    for idx, frame in enumerate(frames):
        ax = axes[idx // cols, idx % cols]
        ax.imshow(frame)
        ax.set_title(f"frame {idx + 1}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_single_sample(
    row: pd.Series,
    config: BaselineConfig,
    model: MultiModalRegressor | None = None,
    metadata_scaler: MetadataScaler | None = None,
    tokenizer: AutoTokenizer | None = None,
    image_processor: AutoImageProcessor | None = None,
) -> None:
    display(Markdown(f"## {row.get('title', row.get('bvid', 'sample'))}"))

    summary = {
        "bvid": row.get("bvid"),
        "owner_name": row.get("owner_name"),
        "tname": row.get("tname"),
        "pubdate": row.get("pubdate"),
        "duration_seconds": row.get("duration_seconds"),
        "stat_view": row.get("stat_view"),
        "log_view": row.get("log_view"),
        "comment_count": row.get("comment_count"),
        "video_path": row.get("video_path"),
        "video_storage_backend": row.get("video_storage_backend"),
    }
    display(pd.Series(summary).to_frame("value"))

    comment_text = str(row.get("comment_text", "无评论"))
    print("Comments:")
    print(textwrap.fill(comment_text, width=100))

    video_path = resolve_video_source(row, config)
    if video_path and Path(video_path).exists():
        try:
            display(Video(video_path, embed=True, html_attributes='controls width="480"'))
        except Exception as exc:
            print(f"无法直接嵌入视频文件，将只展示抽帧结果: {exc}")
        frames = sample_video_frames(video_path, config.clip_seconds, min(config.num_frames, 8), config.image_size)
        show_frame_grid(frames, cols=4, figsize=(16, 6))
    else:
        print("当前样本没有可直接读取的视频文件，先只展示文本和元数据。")

    if model is not None and metadata_scaler is not None and tokenizer is not None and image_processor is not None:
        temp_df = pd.DataFrame([row])
        temp_dataset = BiliPopularityDataset(
            frame=temp_df,
            metadata_scaler=metadata_scaler,
            tokenizer=tokenizer,
            image_processor=image_processor,
            config=config,
        )
        item = temp_dataset[0]
        batch = {
            "pixel_values": item["pixel_values"].unsqueeze(0).to(DEVICE),
            "input_ids": item["input_ids"].unsqueeze(0).to(DEVICE),
            "attention_mask": item["attention_mask"].unsqueeze(0).to(DEVICE),
            "metadata": item["metadata"].unsqueeze(0).to(DEVICE),
        }
        model.eval()
        with torch.no_grad():
            prediction = model(**batch)["prediction"].item()
        print(f"Predicted log1p(view): {prediction:.4f}")
        print(f"Actual log1p(view):    {float(row['log_view']):.4f}")
        print(f"Predicted view:        {float(np.expm1(prediction)):.2f}")
        print(f"Actual view:           {float(row['stat_view']):.2f}")


# 训练完成后，可以直接调用下面这行查看单条样本。
# show_single_sample(splits['test'].iloc[0], config, trained_model, metadata_scaler, tokenizer, image_processor)